<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Lora_fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ---- Training dataset ----
# Format: "Customer: [message] Response: [ideal reply]"
# The model learns to complete the Response: part

from datasets import Dataset

training_examples = [
    "Customer: My order has not arrived yet. Response: Thank you for contacting us. We are sorry to hear your order has not arrived. Please share your order number and we will investigate immediately and update you within 24 hours.",
    "Customer: I want to return this product. Response: We would be happy to help with a return. Our policy allows returns within 30 days. Please ensure the item is in original packaging and we will process your refund within 5 to 7 business days.",
    "Customer: The item I received is damaged. Response: We sincerely apologise for sending a damaged item. Please send a photo of the damage and your order number. We will arrange a replacement or full refund as quickly as possible.",
    "Customer: How long does delivery take? Response: Standard delivery takes 3 to 5 business days. Express delivery takes 1 to 2 business days. You will receive a tracking number by email once your order is dispatched.",
    "Customer: I was charged twice for my order. Response: We apologise for this error. Please share your order number and we will immediately investigate and refund the duplicate charge to your original payment method.",
    "Customer: Can I change my delivery address? Response: Yes, we can update your address if your order has not been dispatched. Please contact us with your order number and new address and we will make the change right away.",
    "Customer: I never received my refund. Response: We are sorry to hear this. Refunds take 5 to 7 business days depending on your bank. Please share your order number and we will check the status immediately.",
    "Customer: The wrong item was delivered. Response: We apologise for this mistake. Please send your order number and a photo of what you received. We will dispatch the correct item right away at no extra cost.",
    "Customer: How do I track my order? Response: You can track your order using the tracking number sent to your email when your order was dispatched. Enter it on our website in the order tracking section.",
    "Customer: I am not happy with my purchase. Response: We are very sorry to hear this. Your satisfaction is our priority. Please tell us what went wrong and we will resolve it with a replacement, refund, or another solution.",
    "Customer: Can I cancel my order? Response: Yes, you can cancel if your order has not been dispatched. Please contact us with your order number. If already shipped, you may return it for a full refund.",
    "Customer: My account has been locked. Response: We are sorry you cannot access your account. Please try resetting your password. If the issue continues contact us and our team will restore access within one business day.",
    "Customer: Do you offer international shipping? Response: Yes, we ship to over 30 countries. International delivery takes 7 to 14 business days. Costs and delivery times will be shown at checkout.",
    "Customer: Can I get a discount on bulk orders? Response: Yes, we offer discounts on orders of 10 units or more. Please contact our sales team with the product and quantity and we will provide a quote within one business day.",
    "Customer: Is this item still in stock? Response: Thank you for your interest. Please share the product name and we will check stock levels. If running low we can notify you as soon as it is available again.",
]

print(f'Training examples: {len(training_examples)}')
print(f'\nFirst example:')
print(f'  {training_examples[0]}')

Training examples: 15

First example:
  Customer: My order has not arrived yet. Response: Thank you for contacting us. We are sorry to hear your order has not arrived. Please share your order number and we will investigate immediately and update you within 24 hours.


In [11]:
# ---- Tokenise the training data ----

MAX_LENGTH = 128

# GPT-2 does not have a pad token by default, so we set it to the EOS token
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    result = tokenizer(
        examples['text'],
        truncation = True,
        max_length = MAX_LENGTH,
        padding    = 'max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

raw_dataset  = Dataset.from_dict({'text': training_examples})
tokenized_ds = raw_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized_ds.set_format('torch')

print(f'Dataset ready: {len(tokenized_ds)} training examples')
print(f'Each example:  {tokenized_ds[0]["input_ids"].shape[0]} tokens')

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Dataset ready: 15 training examples
Each example:  128 tokens


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)
# Total parameters: 124 million (approximately)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 124439808


In [2]:
def ask(model, tokenizer, question, max_new_tokens=30):
    model.eval()  # inference mode, not training mode
    inputs = tokenizer(question, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # deterministic output, temperature = 0
    )
    return tokenizer.decode(outputs[0])

In [3]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                       # rank of the adapter matrices
    lora_alpha=16,             # standard practice: 2 x r
    lora_dropout=0.1,          # regularization to reduce overfitting
    target_modules=["c_attn", "c_proj"],  # apply adapters to GPT-2's attention projection layers
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# trainable params: ~800,000 || all params: ~124,000,000 || trainable%: 0.64

trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="gpt2-support-lora",
    num_train_epochs=20,
    per_device_train_batch_size=8,   # reduced if data or GPU memory is heavy
    learning_rate=1e-3,              # 0.001
    fp16=True,                       # use if a CUDA GPU is available
    logging_steps=1,
)

trainer = Trainer(model=peft_model, args=training_args, train_dataset=tokenized_ds)
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,6.698402
2,5.904490
3,4.749212
4,3.520669
5,1.877184
6,1.190994
7,1.141290
8,1.152152
9,1.117892
10,1.118003


TrainOutput(global_step=40, training_loss=1.3266884744167329, metrics={'train_runtime': 6.6534, 'train_samples_per_second': 45.09, 'train_steps_per_second': 6.012, 'total_flos': 19783758643200.0, 'train_loss': 1.3266884744167329, 'epoch': 20.0})

In [16]:
# ---- Generate responses AFTER fine-tuning ----

print('=== AFTER Fine-Tuning (GPT-2 + LoRA) ===\n')
ft_responses = []
for prompt in test_prompts:
    # Using 'peft_model' and the 'ask' function defined in cell xvdRg6aQSK0B
    resp = ask(peft_model, tokenizer, prompt)
    ft_responses.append(resp)
    print(f'Prompt:   {prompt}')
    print(f'Response: {resp}')
    print()

# Side-by-side comparison
print('=' * 65)
print('BEFORE vs AFTER — Side by Side')
print('=' * 65)
for i, (prompt, before, after) in enumerate(
        zip(test_prompts, baseline_responses, ft_responses), 1):
    question = prompt.replace('Customer:', '').replace('Response:', '').strip()
    print(f'\nQ{i}: {question}')
    print(f'  BEFORE: {before[:110]}')
    print(f'  AFTER:  {after[:110]}')

=== AFTER Fine-Tuning (GPT-2 + LoRA) ===



/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:2944: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Prompt:   Customer: My order has not arrived yet. Response:
Response: Customer: My order has not arrived yet. Response: We apologize for the inconvenience. We will send you a tracking number within 24 hours.<|endoftext|>

Prompt:   Customer: The item I received is damaged. Response:
Response: Customer: The item I received is damaged. Response: We received your item within 24 business days. Please send us a message and we will replace the item.<|endoftext|>

Prompt:   Customer: I want to return this product. Response:
Response: Customer: I want to return this product. Response: We are sorry to hear that you have returned your order. Please contact us within 24 hours.<|endoftext|>

BEFORE vs AFTER — Side by Side


In [17]:
peft_model.save_pretrained("support_adapter")
